# DFI A100 Colab walkthrough

This notebook is a thin orchestrator for the packaged DFI runner. It verifies an exact repository ZIP from Google Drive, extracts it to Colab local SSD, installs the locked GPU environment, runs the real pinned LLaDA model, reuses the Drive-backed inference cache from a fresh process, and reads the sealed outputs.

The checked-in claims are synthetic correctness fixtures. Even on an A100, this run is **not** an audited-anchor result, a completed `DFI-Known` benchmark, or scientific evidence. Every produced receipt must keep `interpretation_allowed=false`.

## 1. Mount Drive and set exact paths

Before running, select an A100 GPU runtime. The repository bundle and its SHA-256 sidecar must already exist at the two literal Drive paths below; this notebook does not search Drive heuristically.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
import hashlib
import json
import os
import re
import shutil
import subprocess
import sys
import zipfile
from pathlib import Path

DRIVE_BUNDLE = Path("/content/drive/MyDrive/DFI/denoising-factual-instability-colab.zip")
DRIVE_BUNDLE_SHA256 = Path(
    "/content/drive/MyDrive/DFI/denoising-factual-instability-colab.zip.sha256"
)
WORK_ROOT = Path("/content/dfi-work")
LOCAL_BUNDLE = Path("/content/denoising-factual-instability-colab.zip")
LOCAL_HF_HOME = Path("/content/huggingface")
RUN_THROUGHPUT_WORKLOAD = False

os.environ["HF_HOME"] = str(LOCAL_HF_HOME)
os.environ["TOKENIZERS_PARALLELISM"] = "false"

In [ ]:
gpu_names = (
    subprocess.check_output(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"], text=True)
    .strip()
    .splitlines()
)
if len(gpu_names) != 1 or "A100" not in gpu_names[0]:
    raise RuntimeError(f"This walkthrough requires one named A100 GPU; found {gpu_names!r}")
print(f"Hardware gate: {gpu_names[0]}")

## 2. Verify the Drive bundle and stage it on local SSD

In [ ]:
for required in (DRIVE_BUNDLE, DRIVE_BUNDLE_SHA256):
    if not required.is_file():
        raise FileNotFoundError(f"Missing required Drive artifact: {required}")
sidecar = DRIVE_BUNDLE_SHA256.read_text(encoding="utf-8").strip().split()
if not sidecar or re.fullmatch(r"[0-9a-f]{64}", sidecar[0]) is None:
    raise ValueError("The Drive SHA-256 sidecar is malformed")
expected_bundle_sha256 = sidecar[0]
shutil.copy2(DRIVE_BUNDLE, LOCAL_BUNDLE)
actual_bundle_sha256 = hashlib.sha256(LOCAL_BUNDLE.read_bytes()).hexdigest()
if actual_bundle_sha256 != expected_bundle_sha256:
    raise ValueError("The local repository ZIP does not match its Drive SHA-256 sidecar")

if Path("/content/dfi-work") != WORK_ROOT:
    raise RuntimeError("Refusing to replace an unexpected working directory")
if WORK_ROOT.exists():
    shutil.rmtree(WORK_ROOT)
WORK_ROOT.mkdir()
with zipfile.ZipFile(LOCAL_BUNDLE) as archive:
    root = WORK_ROOT.resolve()
    for member in archive.infolist():
        target = (WORK_ROOT / member.filename).resolve()
        if not target.is_relative_to(root):
            raise ValueError(f"Unsafe path in repository ZIP: {member.filename!r}")
    archive.extractall(WORK_ROOT)

REPO = WORK_ROOT / "denoising-factual-instability"
if not (REPO / "pyproject.toml").is_file():
    raise FileNotFoundError(f"Expected repository root is missing: {REPO}")
marker = (REPO / ".git_archival.txt").read_text(encoding="utf-8")
match = re.fullmatch(r"commit: ([0-9a-f]{40})\n?", marker)
if match is None:
    raise ValueError("The repository archive has no expanded Git commit marker")
SOURCE_COMMIT = match.group(1)
os.environ["DFI_SOURCE_COMMIT"] = SOURCE_COMMIT
print({"bundle_sha256": actual_bundle_sha256, "source_commit": SOURCE_COMMIT, "repo": str(REPO)})

## 3. Install the locked package and validate CUDA/BF16

The package environment lives under the local-SSD repository. Google Drive is used only for the durable inference-result cache, never for the per-batch GPU hot path.

In [ ]:
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv==0.7.8"], check=True)
subprocess.run(
    ["uv", "sync", "--frozen", "--extra", "gpu", "--no-editable"],
    cwd=REPO,
    check=True,
)
PYTHON = REPO / ".venv" / "bin" / "python"
DFI = REPO / ".venv" / "bin" / "dfi"
cuda_probe = """
import json
import torch
if not torch.cuda.is_available():
    raise RuntimeError('CUDA is unavailable')
if not torch.cuda.is_bf16_supported():
    raise RuntimeError('The selected GPU does not support BF16')
name = torch.cuda.get_device_name(0)
if 'A100' not in name:
    raise RuntimeError(f'Expected A100, found {name}')
print(json.dumps({
    'torch': torch.__version__,
    'cuda': torch.version.cuda,
    'gpu': name,
    'bf16': True,
}))
"""
subprocess.run([PYTHON, "-c", cuda_probe], cwd=REPO, check=True)

In [ ]:
subprocess.run([DFI, "check"], cwd=REPO, check=True)

## 4. Run the real pinned model experiment

This invocation scores 8 synthetic claims × 32 masks (256 logical requests). If a valid cache already exists, the workload may be hydrated instead of recomputed; the receipt makes that visible. The separate manual scalar/global parity acceptance gate is intentionally not a prerequisite for running this demonstration.

In [ ]:
OUTPUT_ROOT = REPO / "runs"
OUTPUT_ROOT.mkdir(exist_ok=True)


def run_dfi(config_name, *, parity=False):
    before = {path.resolve() for path in OUTPUT_ROOT.iterdir() if path.is_dir()}
    command = [DFI, "run", REPO / "configs" / config_name]
    if parity:
        command.append("--parity")
    subprocess.run(command, cwd=REPO, check=True, env=os.environ.copy())
    completed = [
        path.resolve()
        for path in OUTPUT_ROOT.iterdir()
        if path.is_dir() and path.resolve() not in before and (path / "run.json").is_file()
    ]
    if len(completed) != 1:
        raise RuntimeError(f"Expected one new sealed run, found {completed!r}")
    return completed[0]


PRIMARY_RUN = run_dfi("a100-smoke.toml", parity=False)
PRIMARY_RECEIPT = json.loads((PRIMARY_RUN / "run.json").read_text(encoding="utf-8"))
if PRIMARY_RECEIPT["interpretation_allowed"] is not False:
    raise RuntimeError("Synthetic run unexpectedly permits interpretation")
print(PRIMARY_RUN)

In [ ]:
primary_summary = {
    "run_uuid": PRIMARY_RECEIPT["run_uuid"],
    "source_commit": PRIMARY_RECEIPT["git"]["commit"],
    "device_name": PRIMARY_RECEIPT["environment"]["device_name"],
    "logical_requests": PRIMARY_RECEIPT["requests"]["logical_requests"],
    "unique_requests": PRIMARY_RECEIPT["requests"]["unique_requests"],
    "inference_forwards": PRIMARY_RECEIPT["requests"]["inference_forwards"],
    "acceptance_forwards": PRIMARY_RECEIPT["requests"]["acceptance_forwards"],
    "scalar_batch_parity": PRIMARY_RECEIPT["acceptance"]["scalar_batch_parity"]["status"],
    "cache_hits": PRIMARY_RECEIPT["cache"]["hits"],
    "cache_misses": PRIMARY_RECEIPT["cache"]["misses"],
    "peak_vram_bytes": PRIMARY_RECEIPT["performance"]["peak_vram_bytes"],
    "padding_ratio": PRIMARY_RECEIPT["batching"]["padding_ratio"],
    "interpretation_allowed": PRIMARY_RECEIPT["interpretation_allowed"],
}
primary_summary

## 5. Start a fresh process and require zero covered model forwards

This invocation omits `--parity`, creates a fresh local run directory, hydrates matching immutable shards from Drive, and must perform zero inference forwards for the covered workload. Model loading and cheap deterministic planning still occur.

In [ ]:
WARM_RUN = run_dfi("a100-smoke.toml", parity=False)
WARM_RECEIPT = json.loads((WARM_RUN / "run.json").read_text(encoding="utf-8"))
if WARM_RECEIPT["requests"]["inference_forwards"] != 0:
    raise RuntimeError("Warm Drive-cache replay performed covered model forwards")
if WARM_RECEIPT["cache"]["misses"] != 0:
    raise RuntimeError("Warm Drive-cache replay reported cache misses")
if WARM_RECEIPT["results"]["sha256"] != PRIMARY_RECEIPT["results"]["sha256"]:
    raise RuntimeError("Warm replay changed the sealed result bytes")
{
    "warm_run": str(WARM_RUN),
    "cache_hits": WARM_RECEIPT["cache"]["hits"],
    "cache_misses": WARM_RECEIPT["cache"]["misses"],
    "inference_forwards": WARM_RECEIPT["requests"]["inference_forwards"],
    "result_sha256": WARM_RECEIPT["results"]["sha256"],
}

## 6. Recompute evaluation from sealed sufficient statistics and inspect rows

In [ ]:
evaluation_text = subprocess.check_output(
    [DFI, "evaluate", WARM_RUN], cwd=REPO, text=True, env=os.environ.copy()
)
EVALUATION = json.loads(evaluation_text)
if EVALUATION["interpretation_allowed"] is not False:
    raise RuntimeError("Synthetic evaluation unexpectedly permits interpretation")
{
    "n_mask_rows": EVALUATION["n_mask_rows"],
    "n_claim_rows": EVALUATION["n_claim_rows"],
    "arms": list(EVALUATION["arms"]),
    "warning": EVALUATION["warning"],
}

In [ ]:
row_probe = """
import json
import sys
from dfi.pipeline import read_parquet_rows
print(json.dumps(read_parquet_rows(sys.argv[1])[:3], sort_keys=True))
"""
row_text = subprocess.check_output(
    [PYTHON, "-c", row_probe, WARM_RUN / "results.parquet"],
    cwd=REPO,
    text=True,
    env=os.environ.copy(),
)
json.loads(row_text)

## 7. Optional 4,096-request performance workload

Set `RUN_THROUGHPUT_WORKLOAD=True` in the settings cell to execute the separate performance-only plan. One execution is **not** the blueprint's accepted baseline: that still requires five warm-up batches and three complete timed repetitions on the named A100.

In [ ]:
if RUN_THROUGHPUT_WORKLOAD:
    THROUGHPUT_RUN = run_dfi("a100-throughput.toml", parity=False)
    THROUGHPUT_RECEIPT = json.loads((THROUGHPUT_RUN / "run.json").read_text(encoding="utf-8"))
    if THROUGHPUT_RECEIPT["requests"]["unique_requests"] != 4096:
        raise RuntimeError("The fixed throughput plan did not contain 4,096 unique requests")
    print(
        {
            "run": str(THROUGHPUT_RUN),
            "unique_requests": THROUGHPUT_RECEIPT["requests"]["unique_requests"],
            "active_wall_seconds": THROUGHPUT_RECEIPT["performance"]["active_wall_seconds"],
            "computed_requests_per_second": THROUGHPUT_RECEIPT["performance"][
                "computed_requests_per_second"
            ],
            "peak_vram_bytes": THROUGHPUT_RECEIPT["performance"]["peak_vram_bytes"],
            "interpretation_allowed": THROUGHPUT_RECEIPT["interpretation_allowed"],
        }
    )
else:
    print("Skipped the optional 4,096-request workload.")

## Evidence boundary

A completed notebook establishes only what its validated receipts show. The default cells can establish a real pinned-model execution smoke, exact result cardinality, and zero-forward reuse from the Drive cache. Scalar/global BF16 parity remains a separate manual A100 acceptance gate and is not established by this walkthrough. The notebook also does not establish the historical audited-anchor gate, the three-repetition performance gate, corruption recovery unless that separate test is run, or any scientific claim.